<h2>Tiered Budget System</h2>

<table border="1" cellpadding="8" cellspacing="0" style="border-collapse: collapse; text-align: center;">
  <tr style="background-color: #bb5959;">
    <th>Tier</th>
    <th>Max Budget</th>
    <th>Time Range</th>
    <th>Slots</th>
  </tr>
  <tr><td>0 (Highest)</td><td>795s</td><td>500-795s</td><td>3</td></tr>
  <tr><td>1</td><td>500s</td><td>295-500s</td><td>10</td></tr>
  <tr><td>2</td><td>295s</td><td>273-295s</td><td>7</td></tr>
  <tr><td>Base</td><td>273s</td><td>≤273s</td><td>remaining</td></tr>
</table>

<h4>How it works:</h4>
<ul>
  <li><b>High budget</b> = fixed at Tier 0 max_budget (795s) — always offer maximum</li>
  <li><b>Reserved time</b> = distribute remaining problems across tiers highest→lowest (worst-case), capped by available slots</li>
  <li><b>Budget given</b> = min(time_left - reserved_time, high)</li>
  <li><b>Slot consumed</b> = based on <i>actual time used</i>, not budget given</li>
  <li><b>Pre-decrement assumption</b>: Current problem assumed to use highest available tier for reserved_time calc</li>
  <li><b>Overflow rule</b>: If actual_time > 795s (timing overhead) → targets Tier 0</li>
  <li><b>Cascade UP</b>: If target tier full → borrow from first available upper tier (conservative)</li>
  <li><b>Cascade DOWN</b>: If no upper tier available → borrow from highest available lower tier</li>
</ul>

<h4>Reserved Time Example (10 problems left, slots [3,10,7], easy first scenario):</h4>
<pre>
Current assumed → Tier 0 (virtually: [2,10,7])
Distribute 9 remaining problems across tiers highest→lowest:
  Tier 0: min(2, 9) = 2 assigned, 7 remaining
  Tier 1: min(10, 7) = 7 assigned, 0 remaining
Reserved = 2×795 + 7×500 = 5,090s
Budget = time_left - 5090, capped at 795s
→ Saved time from easy problems is fully available for hard ones!
</pre>

<p><i>Reserved time is never more than what remaining problems could actually need — no phantom slot reservation.</i></p>

In [1]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

Found existing installation: keras 3.10.0
Uninstalling keras-3.10.0:
  Successfully uninstalled keras-3.10.0
Found existing installation: matplotlib 3.10.0
Uninstalling matplotlib-3.10.0:
  Successfully uninstalled matplotlib-3.10.0
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
Found existing installation: tensorflow 2.19.0
Uninstalling tensorflow-2.19.0:
  Successfully uninstalled tensorflow-2.19.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
warnings.simplefilter('ignore')

In [3]:
import os
import sys
import subprocess

In [4]:
def set_env(input_archive, temp_dir):

    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir, exist_ok=True)
        
        subprocess.run(['tar', '-xzf', input_archive, '-C', temp_dir], check=True)
    
    subprocess.run([
        sys.executable, 
        '-m', 
        'pip', 
        'install', 
        '--no-index', 
        '--find-links', 
        f'{temp_dir}/wheels', 
        'unsloth', 
        'trl', 
        'vllm', 
        'openai_harmony'
    ], check=True)

In [5]:
set_env(
    input_archive='/kaggle/input/aimo-3-utils/wheels.tar.gz', 
    temp_dir='/kaggle/tmp/setup'
)

Looking in links: /kaggle/tmp/setup/wheels
Processing /kaggle/tmp/setup/wheels/unsloth-2025.12.9-py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/trl-0.24.0-py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/vllm-0.11.2-cp38-abi3-manylinux1_x86_64.whl
Processing /kaggle/tmp/setup/wheels/openai_harmony-0.0.8-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Processing /kaggle/tmp/setup/wheels/unsloth_zoo-2025.12.7-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/tyro-1.0.3-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/xformers-0.0.33.post1-cp39-abi3-manylinux_2_28_x86_64.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/bitsandbytes-0.49.0-py3-none-manylinux_2_24_x86_64.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/datasets-4.3.0-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/prometheus_fastapi_instrumentator-7.1.0-py3-none-any.whl (from vllm)
Processing /kaggle/tmp/setup/wheels/lm_format_enforcer-0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kauldron 1.3.0 requires scikit-learn, which is not installed.
kauldron 1.3.0 requires tensorflow, which is not installed.
ydata-profiling 4.18.0 requires matplotlib<=3.10,>=3.5, which is not installed.
pyldavis 3.4.1 requires scikit-learn>=1.0.0, which is not installed.
stable-baselines3 2.1.0 requires matplotlib, which is not installed.
sentence-transformers 5.1.1 requires scikit-learn, which is not installed.
librosa 0.11.0 requires scikit-learn>=1.1.0, which is not installed.
cuml-cu12 25.6.0 requires scikit-learn>=1.5, which is not installed.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
bigframes 2.26.0 requires matplotlib>=3.7.1, which is not installed.
arviz 0.22.0 requires matplotlib>=3.8, which is not installed.
pynndescent 0.5.13 requires scikit-learn>=0.

In [6]:
subprocess.run(['ls', '/kaggle/tmp/setup/tiktoken_encodings'])

cl100k_base.tiktoken
o200k_base.tiktoken


CompletedProcess(args=['ls', '/kaggle/tmp/setup/tiktoken_encodings'], returncode=0)

In [7]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['TIKTOKEN_ENCODINGS_BASE'] = '/kaggle/tmp/setup/tiktoken_encodings'

In [8]:
import gc
import re
import math
import time
import queue
import threading
import contextlib
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import numpy as np
import pandas as pd
import polars as pl

# Configure matplotlib before importing pyplot
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['Liberation Sans', 'DejaVu Sans', 'Arial', 'Helvetica', 'sans-serif']
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

import matplotlib.pyplot as plt
import matplotlib.cm as cm

from openai import OpenAI

from openai_harmony import (
    HarmonyEncodingName, 
    load_harmony_encoding, 
    SystemContent, 
    ReasoningEffort, 
    ToolNamespaceConfig, 
    Author, 
    Message, 
    Role, 
    TextContent, 
    Conversation
)

from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server

In [9]:
class CFG:
    
    system_prompt = (
        'You are an elite mathematical problem solver with expertise at the International '
        'Mathematical Olympiad (IMO) level. Your goal is to find the correct answer through '
        'rigorous mathematical reasoning.\n\n'
        
        '# Problem-Solving Approach:\n'
        '1. UNDERSTAND: Carefully read and rephrase the problem in your own words. '
        'Identify what is given, what needs to be found, and any constraints.\n'
        '2. EXPLORE: Consider multiple solution strategies. Think about relevant theorems, '
        'techniques, patterns, or analogous problems. Don\'t commit to one approach immediately.\n'
        '3. PLAN: Select the most promising approach and outline key steps before executing.\n'
        '4. EXECUTE: Work through your solution methodically. Show all reasoning steps clearly.\n'
        '5. VERIFY: Check your answer by substituting back, testing edge cases, or using '
        'alternative methods. Ensure logical consistency throughout.\n\n'
        
        '# Mathematical Reasoning Principles:\n'
        '- Break complex problems into smaller, manageable sub-problems\n'
        '- Look for patterns, symmetries, and special cases that provide insight\n'
        '- Use concrete examples to build intuition before generalizing\n'
        '- Consider extreme cases and boundary conditions\n'
        '- If stuck, try working backwards from the desired result\n'
        '- Be willing to restart with a different approach if needed\n\n'
        
        '# Verification Requirements:\n'
        '- Cross-check arithmetic and algebraic manipulations\n'
        '- Verify that your solution satisfies all problem constraints\n'
        '- Test your answer with simple cases or special values when possible\n'
        '- Ensure dimensional consistency and reasonableness of the result\n\n'
        
        '# Output Format:\n'
        'The final answer must be a non-negative integer between 0 and 99999.\n'
        'Place your final numerical answer inside \\boxed{}, e.g., \\boxed{42}\n\n'
        
        'Think step-by-step and show your complete reasoning process. Quality of reasoning '
        'is as important as the final answer.'
    )
    
    tool_prompt = (
        'Use this tool to execute Python code for:\n'
        '- Complex calculations that would be error-prone by hand\n'
        '- Numerical verification of analytical results\n'
        '- Generating examples or testing conjectures\n'
        '- Visualizing problem structure when helpful\n'
        '- Brute-force verification for small cases\n\n'
        
        'The environment is a stateful Jupyter notebook. Code persists between executions.\n'
        'Always use print() to display results. Write clear, well-commented code.\n\n'
        
        'Remember: Code should support your mathematical reasoning, not replace it. '
        'Explain what you\'re computing and why before running code.'
    )
    
    preference_prompt = (
        'You have access to `math`, `numpy`, `sympy`, `itertools`, and `mpmath`.\n\n'
        
        '# Tool Selection Guide:\n'
        '- mpmath: CRITICAL for geometry, trigonometry, and logarithms. The environment is pre-configured '
        'with `mpmath.mp.dps = 64` (64 decimal places of precision). Always use this instead of standard '
        'floats to avoid rounding errors on final integer extraction.\n'
        '- sympy: Best for exact symbolic answers, equation solving, diophantine equations, '
        'and prime factorization.\n'
        '- itertools: Crucial for combinatorics, permutations, and state-space searches.\n'
        '- numpy: Best for matrix operations and linear algebra.\n\n'
        
        'Combine approaches: Derive a formula symbolically with `sympy`, then verify it '
        'numerically to 64 decimals using `mpmath`.'
    )

    served_model_name = 'gpt-oss'
    model_path = '/kaggle/input/gpt-oss-120b/transformers/default/1'
    
    kv_cache_dtype = 'fp8_e4m3'
    dtype = 'auto'

    # Tiered budget system: (max_budget, lower_bound, slots)
    # A problem consuming time in range (lower_bound, max_budget] uses a slot from this tier
    budget_tiers = [
        {'max_budget': 820, 'lower_bound': 550, 'slots': 5},   # 3 problems can use 500-795s
        {'max_budget': 550, 'lower_bound': 300, 'slots': 9},  # 10 problems can use 295-500s
        {'max_budget': 350, 'lower_bound': 275, 'slots': 6},   # 7 problems can use 273-295s
        {'max_budget': 275, 'lower_bound': 200, 'slots': 6},   # 8 problems can use 200-273s
    ]

    base_problem_timeout = 200  # Base timeout for remaining problems

    notebook_limit = 17600 # if os.getenv('KAGGLE_IS_COMPETITION_RERUN') else 9800 # 9800 for 28 prblms
    server_timeout = 180

    session_timeout = 960
    jupyter_timeout = 10
    sandbox_timeout = 5

    stream_interval = 200
    context_tokens = 65536
    search_tokens = 1024
    buffer_tokens = 512
    batch_size = 256
    early_stop = 8
    attempts = 8  
    workers = 16
    turns = 128
    seed = 42

    gpu_memory_utilization = 0.96
    temperature = 1.0
    min_p = 0.02

In [10]:
set_seed(CFG.seed)

In [11]:
class AIMO3Template:

    def __init__(self):

        pass

    def get_system_content(self, system_prompt: str, tool_config: ToolNamespaceConfig) -> SystemContent:

        return (
            SystemContent.new()
            .with_model_identity(system_prompt)
            .with_reasoning_effort(reasoning_effort=ReasoningEffort.HIGH)
            .with_tools(tool_config)
        )

    def apply_chat_template(
        self, 
        system_prompt: str, 
        user_prompt: str, 
        tool_config: ToolNamespaceConfig
    ) -> list[Message]:

        system_content = self.get_system_content(system_prompt, tool_config)        
        system_message = Message.from_role_and_content(Role.SYSTEM, system_content)

        user_message = Message.from_role_and_content(Role.USER, user_prompt)

        return [system_message, user_message]

In [12]:
class AIMO3Sandbox:

    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:

        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count

            return ports

    def __init__(self, timeout: float):

        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None
        
        ports = self._get_next_ports(5)

        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'

        self._km = KernelManager()
        self._km.shell_port = ports[0]
        self._km.iopub_port = ports[1]
        self._km.stdin_port = ports[2]
        self._km.hb_port = ports[3]
        self._km.control_port = ports[4]

        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])

        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True

        self.execute(
            'import math\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import numpy as np\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def _format_error(self, traceback: list[str]) -> str:

        clean_lines = []

        for frame in traceback:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)

            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue

            clean_lines.append(clean_frame)

        return ''.join(clean_lines)

    def execute(self, code: str, timeout: float | None = None) -> str:

        client = self._client
        effective_timeout = timeout or self._default_timeout
        
        msg_id = client.execute(
            code, 
            store_history=True, 
            allow_stdin=False, 
            stop_on_error=False
        )

        stdout_parts = []
        stderr_parts = []
        
        start_time = time.time()

        while True:
            elapsed = time.time() - start_time

            if elapsed > effective_timeout:
                self._km.interrupt_kernel()

                return f'[ERROR] Execution timed out after {effective_timeout} seconds'

            try:
                msg = client.get_iopub_msg(timeout=1.0)

            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue

            msg_type = msg.get('msg_type')
            content = msg.get('content', {})

            if msg_type == 'stream':
                text = content.get('text', '')

                if content.get('name') == 'stdout':
                    stdout_parts.append(text)

                else:
                    stderr_parts.append(text)

            elif msg_type == 'error':
                traceback_list = content.get('traceback', [])

                stderr_parts.append(self._format_error(traceback_list))

            elif msg_type in {'execute_result', 'display_data'}:
                data = content.get('data', {})
                text = data.get('text/plain')

                if text:
                    stdout_parts.append(text if text.endswith('\n') else f'{text}\n')

            elif msg_type == 'status':
                if content.get('execution_state') == 'idle':
                    break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)

        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr

        return stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'

    def close(self):

        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()

        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)

            with contextlib.suppress(Exception):
                self._km.cleanup_resources()

    def reset(self):

        self.execute('%reset -f')
        self.execute('import gc; gc.collect()')

        self.execute(
            'import math\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import numpy as np\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def __del__(self):

        self.close()

In [13]:
class AIMO3Tool:

    def __init__(self, local_jupyter_timeout: float, tool_prompt: str, sandbox=None):

        self._local_jupyter_timeout = local_jupyter_timeout
        self._tool_prompt = tool_prompt
        self._jupyter_session = sandbox
        
        self._owns_session = sandbox is None
        
        self._execution_lock = threading.Lock()
        self._init_lock = threading.Lock()

    def _ensure_session(self):

        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)

    def _ensure_last_print(self, code: str) -> str:

        lines = code.strip().split('\n')

        if not lines:
            return code

        last_line = lines[-1].strip()

        if 'print' in last_line or 'import' in last_line:
            return code

        if not last_line:
            return code

        if last_line.startswith('#'):
            return code

        lines[-1] = 'print(' + last_line + ')'

        return '\n'.join(lines)

    @property
    def instruction(self) -> str:

        return self._tool_prompt

    @property
    def tool_config(self) -> ToolNamespaceConfig:

        return ToolNamespaceConfig(
            name='python', 
            description=self.instruction, 
            tools=[]
        )

    def _make_response(self, output: str, channel: str | None = None) -> Message:

        content = TextContent(text=output)
        author = Author(role=Role.TOOL, name='python')
        message = Message(author=author, content=[content]).with_recipient('assistant')

        if channel:
            message = message.with_channel(channel)

        return message

    def process_sync_plus(self, message: Message) -> list[Message]:

        self._ensure_session()
        raw_script = message.content[0].text
        final_script = self._ensure_last_print(raw_script)

        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script)

            except TimeoutError as exc:
                output = f'[ERROR] {exc}'

        return [self._make_response(output, channel=message.channel)]

    def close(self):

        if self._jupyter_session is not None:
            if self._owns_session:
                self._jupyter_session.close()

            self._jupyter_session = None

    def __del__(self):

        self.close()

In [14]:
class AIMO3Solver:

    def __init__(self, cfg, port: int = 8000):

        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'
        self.template = AIMO3Template()
        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()

        self._preload_model_weights()
        
        self.server_process = self._start_server()

        self.client = OpenAI(
            base_url=self.base_url, 
            api_key=self.api_key, 
            timeout=self.cfg.session_timeout
        )

        self._wait_for_server()
        self._initialize_kernels()

        self.notebook_start_time = time.time()
        self.problems_remaining = 50 if os.getenv('KAGGLE_IS_COMPETITION_RERUN') else 28
        self.problem_counter = 0  # For naming plot files
        
        # Initialize tiered budget tracking
        # Each tier tracks remaining slots for problems that use time in (lower_bound, max_budget]
        self.tier_slots_remaining = [tier['slots'] for tier in self.cfg.budget_tiers]
        self.problem_times = []  # Track actual time used by each problem for debugging

    def _preload_model_weights(self) -> None:

        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start_time = time.time()
        
        files_to_load = []
        total_size = 0

        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)

                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)

        def _read_file(path: str) -> None:

            with open(path, 'rb') as file_object:
                while file_object.read(1024 * 1024 * 1024):
                    pass

        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            list(executor.map(_read_file, files_to_load))

        elapsed = time.time() - start_time
        print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')

    def _start_server(self) -> subprocess.Popen:

        cmd = [
            sys.executable, 
            '-m', 
            'vllm.entrypoints.openai.api_server', 
            '--seed', 
            str(self.cfg.seed), 
            '--model', 
            self.cfg.model_path, 
            '--served-model-name', 
            self.cfg.served_model_name, 
            '--tensor-parallel-size', 
            '1', 
            '--max-num-seqs', 
            str(self.cfg.batch_size), 
            '--gpu-memory-utilization', 
            str(self.cfg.gpu_memory_utilization), 
            '--host', 
            '0.0.0.0', 
            '--port', 
            str(self.port), 
            '--dtype', 
            self.cfg.dtype, 
            '--kv-cache-dtype', 
            self.cfg.kv_cache_dtype, 
            '--max-model-len', 
            str(self.cfg.context_tokens), 
            '--stream-interval', 
            str(self.cfg.stream_interval), 
            '--async-scheduling', 
            '--enable-prefix-caching'
        ]

        self.log_file = open('vllm_server.log', 'w')

        return subprocess.Popen(
            cmd, 
            stdout=self.log_file, 
            stderr=subprocess.STDOUT, 
            start_new_session=True
        )

    def _wait_for_server(self):

        print('Waiting for vLLM server...')
        start_time = time.time()

        for _ in range(self.cfg.server_timeout):
            return_code = self.server_process.poll()

            if return_code is not None:
                self.log_file.flush()

                with open('vllm_server.log', 'r') as log_file:
                    logs = log_file.read()

                raise RuntimeError(f'Server died with code {return_code}. Full logs:\n{logs}\n')

            try:
                self.client.models.list()
                elapsed = time.time() - start_time
                print(f'Server is ready (took {elapsed:.2f} seconds).\n')

                return

            except Exception:
                time.sleep(1)

        raise RuntimeError('Server failed to start (timeout).\n')

    def _initialize_kernels(self) -> None:

        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels...')
        start_time = time.time()

        self.sandbox_pool = queue.Queue()

        def _create_sandbox():
            
            return AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)

        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures = [executor.submit(_create_sandbox) for _ in range(self.cfg.workers)]

            for future in as_completed(futures):
                self.sandbox_pool.put(future.result())

        elapsed = time.time() - start_time
        print(f'Kernels initialized in {elapsed:.2f} seconds.\n')

    def _calculate_reserved_time(self, problems_left_others: int) -> float:
        """
        Calculate reserved time for future problems using tiered budget system.
        
        Assumes the CURRENT problem will use highest available tier.
        For reserved time calculation, we virtually pre-decrement from highest available tier.
        
        Distributes problems_left_others across tiers from highest to lowest
        (worst-case: future problems fill expensive tiers first), capped by
        available virtual slots per tier. Remaining problems use base timeout.
        """
        # Virtual slots: assume current problem uses highest available tier
        virtual_slots = self.tier_slots_remaining.copy()
        
        # Virtually decrement from highest available tier
        for i in range(len(virtual_slots)):
            if virtual_slots[i] > 0:
                virtual_slots[i] -= 1
                break
        
        # Distribute remaining problems across tiers from highest to lowest
        # (worst-case assumption: future problems use the most expensive tiers first)
        # Cap each tier's contribution at its available slots AND remaining problems
        reserved = 0.0
        remaining_to_assign = problems_left_others
        
        for i, tier in enumerate(self.cfg.budget_tiers):
            assigned = min(virtual_slots[i], remaining_to_assign)
            reserved += assigned * tier['max_budget']
            remaining_to_assign -= assigned
            if remaining_to_assign <= 0:
                break
        
        # Any leftover problems use base timeout
        if remaining_to_assign > 0:
            reserved += remaining_to_assign * self.cfg.base_problem_timeout
        
        return reserved

    def _record_time_usage(self, actual_time: float) -> None:
        """
        Record which tier a problem's actual time usage falls into.
        
        Finds the appropriate tier based on actual time spent and decrements
        that tier's remaining slots. This allows problems that finish quickly
        to not "waste" high-tier slots.
        
        Tier assignment:
        - If time > tier_0_max_budget (overflow) → target tier 0
        - If time > tier[i].lower_bound and time <= tier[i].max_budget → target tier i
        - If time <= base_problem_timeout → no tier slot consumed (base pool)
        
        Cascade logic:
        - Try target tier first
        - If full → cascade UPWARD (borrow from higher tier, conservative)
        - If no upper tier available → cascade DOWNWARD (borrow from lower tier)
        - If ALL tiers exhausted → budget overrun warning (no slot consumed)
        """
        self.problem_times.append(actual_time)
        
        target_tier = None
        
        # Overflow: actual_time exceeds Tier 0's max_budget (timing overhead)
        if actual_time > self.cfg.budget_tiers[0]['max_budget']:
            target_tier = 0
        else:
            # Normal case: find which tier this time falls into (highest to lowest)
            for i, tier in enumerate(self.cfg.budget_tiers):
                lower = tier['lower_bound']
                upper = tier['max_budget']
                
                if lower < actual_time <= upper:
                    target_tier = i
                    break
        
        # If time is at or below base - no tier slot consumed
        if target_tier is None:
            print(f"⏱️ Time {actual_time:.1f}s is in base tier (≤{self.cfg.base_problem_timeout}s) | No slot consumed")
            print(f"   Tier slots remaining: {self.tier_slots_remaining}")
            return
        
        # Try to consume a slot from the target tier
        tier = self.cfg.budget_tiers[target_tier]
        lower, upper = tier['lower_bound'], tier['max_budget']
        
        if self.tier_slots_remaining[target_tier] > 0:
            self.tier_slots_remaining[target_tier] -= 1
            tier_info = f"Tier {target_tier} ({lower}-{upper}s)"
            print(f"⏱️ Time {actual_time:.1f}s assigned to {tier_info} | Slots remaining: {self.tier_slots_remaining}")
        else:
            # Target tier is full - try cascade
            borrowed = False
            
            # 1. CASCADE UPWARD: borrow from first available upper tier (conservative)
            for upper_tier in range(target_tier - 1, -1, -1):
                if self.tier_slots_remaining[upper_tier] > 0:
                    self.tier_slots_remaining[upper_tier] -= 1
                    info = self.cfg.budget_tiers[upper_tier]
                    print(f"⚠️ Tier {target_tier} ({lower}-{upper}s) full | Time {actual_time:.1f}s borrows UP from Tier {upper_tier} ({info['lower_bound']}-{info['max_budget']}s)")
                    print(f"   Slots remaining: {self.tier_slots_remaining}")
                    borrowed = True
                    break
            
            # 2. CASCADE DOWNWARD: if no upper tier available, borrow from highest available lower tier
            if not borrowed:
                for lower_tier in range(target_tier + 1, len(self.tier_slots_remaining)):
                    if self.tier_slots_remaining[lower_tier] > 0:
                        self.tier_slots_remaining[lower_tier] -= 1
                        info = self.cfg.budget_tiers[lower_tier]
                        print(f"⚠️ Tier {target_tier} ({lower}-{upper}s) full | Time {actual_time:.1f}s borrows DOWN from Tier {lower_tier} ({info['lower_bound']}-{info['max_budget']}s)")
                        print(f"   Slots remaining: {self.tier_slots_remaining}")
                        borrowed = True
                        break
            
            if not borrowed:
                # ALL tiers exhausted - true budget overrun
                print(f"🚨 BUDGET OVERRUN: Time {actual_time:.1f}s in Tier {target_tier} range, ALL tier slots exhausted!")
                print(f"   Slots remaining: {self.tier_slots_remaining}")

    def _scan_for_answer(self, text: str) -> int | None:

        pattern = r'\\boxed\s*\{\s*([0-9,]+)\s*\}'
        matches = re.findall(pattern, text)

        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)

                if 0 <= value <= 99999:
                    return value

            except ValueError:
                pass

        return None

    def _process_attempt(
        self, 
        problem: str, 
        system_prompt: str, 
        attempt_index: int, 
        stop_event: threading.Event, 
        deadline: float
    ) -> dict:

        if stop_event.is_set() or time.time() > deadline:
            return {
                'Attempt': attempt_index + 1, 
                'Answer': None, 
                'Python Calls': 0, 
                'Python Errors': 0, 
                'Response Length': 0
            }

        local_tool = None
        sandbox = None
        python_calls = 0
        python_errors = 0
        total_tokens = 0
        final_answer = None

        attempt_seed = int(math.pow(self.cfg.seed + attempt_index, 2))

        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)

            local_tool = AIMO3Tool(
                local_jupyter_timeout=self.cfg.jupyter_timeout, 
                tool_prompt=self.cfg.tool_prompt, 
                sandbox=sandbox
            )

            encoding = self.encoding
            messages = self.template.apply_chat_template(
                system_prompt, 
                problem, 
                local_tool.tool_config
            )

            conversation = Conversation.from_messages(messages)

            for _ in range(self.cfg.turns):
                if stop_event.is_set() or time.time() > deadline:
                    break

                prompt_ids = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
                max_tokens = self.cfg.context_tokens - len(prompt_ids)

                if max_tokens < self.cfg.buffer_tokens:
                    break

                stream = None
                try:
                    stream = self.client.completions.create(
                        model=self.cfg.served_model_name, 
                        temperature=self.cfg.temperature, 
                        max_tokens=max_tokens, 
                        prompt=prompt_ids, 
                        seed=attempt_seed, 
                        stream=True, 
                        extra_body={
                            'min_p': self.cfg.min_p, 
                            'stop_token_ids': self.stop_token_ids, 
                            'return_token_ids': True
                        },
                        timeout=max(0, deadline - time.time()),
                    )
                except Exception as e:
                    print(f"⚠️ Failed to create completion stream: {e}")
                    break
    
                if stream is None:
                    continue

                try:
                    token_buffer = []
                    text_chunks = []

                    for chunk in stream:
                        if stop_event.is_set() or time.time() > deadline:
                            break

                        new_tokens = chunk.choices[0].token_ids
                        new_text = chunk.choices[0].text

                        if new_tokens:
                            token_buffer.extend(new_tokens)
                            total_tokens += len(new_tokens)
                            text_chunks.append(new_text)

                        if '}' in new_text:
                            search_text = ''.join(text_chunks[-self.cfg.search_tokens:])
                            answer = self._scan_for_answer(search_text)

                            if answer is not None:
                                final_answer = answer
                                break

                finally:
                    stream.close()


                if final_answer is not None:
                    break

                if not token_buffer:
                    break

                new_messages = encoding.parse_messages_from_completion_tokens(token_buffer, Role.ASSISTANT)
                conversation.messages.extend(new_messages)
                last_message = new_messages[-1]

                if last_message.channel == 'final':
                    answer_text = last_message.content[0].text
                    final_answer = self._scan_for_answer(answer_text)
                    break

                if last_message.recipient == 'python':
                    python_calls += 1
                    python_code = last_message.content[0].text
                    
                    print("🐍 Executing Python code...")
                    tool_responses = local_tool.process_sync_plus(last_message)

                    response_text = tool_responses[0].content[0].text

                    if response_text.startswith('[ERROR]') or 'Traceback' in response_text or 'Error:' in response_text:
                        python_errors += 1

                    conversation.messages.extend(tool_responses)

        except Exception as exc:
            python_errors += 1

        finally:
            if local_tool is not None:
                local_tool.close()

            if sandbox is not None:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)


        return {
            'Attempt': attempt_index + 1, 
            'Response Length': total_tokens, 
            'Python Calls': python_calls, 
            'Python Errors': python_errors, 
            'Answer': final_answer
        }

    def _select_answer(self, detailed_results: list) -> int:
        """
        Simple majority voting with error ratio tie-breaker.
        """
        # Filter to only traces with valid answers
        valid_results = [r for r in detailed_results if r['Answer'] is not None]
        
        if not valid_results:
            print('\nNo valid answers found.')
            return 0
        
        answer_counts = defaultdict(int)
        answer_calls = defaultdict(int)
        answer_errors = defaultdict(int)

        for result in valid_results:
            answer = result['Answer']

            answer_counts[answer] += 1
            answer_calls[answer] += result['Python Calls']
            answer_errors[answer] += result['Python Errors']

        answer_stats = {}
        for ans in answer_counts:
            calls = answer_calls[ans]
            errors = answer_errors[ans]
            error_ratio = errors / calls if calls > 0 else 0.0
            answer_stats[ans] = {
                'count': answer_counts[ans],
                'calls': calls,
                'errors': errors,
                'ratio': error_ratio
            }

        # Sort by count (desc), then by error ratio (asc)
        sorted_answers = sorted(
            answer_counts.keys(), 
            key=lambda ans: (answer_stats[ans]['count'], -answer_stats[ans]['ratio']), 
            reverse=True
        )

        vote_data = []
        for answer in sorted_answers:
            stats = answer_stats[answer]
            vote_data.append((
                answer, 
                stats['count'],
                stats['calls'],
                stats['errors'],
                round(stats['ratio'], 4)
            ))

        vote_dataframe = pd.DataFrame(vote_data, columns=['Answer', 'Votes', 'Total Calls', 'Total Errors', 'Error Ratio'])
        display(vote_dataframe)

        final_answer = sorted_answers[0]
        final_votes = answer_stats[final_answer]['count']
        final_ratio = answer_stats[final_answer]['ratio']

        print(f'\nFinal Result: {final_answer} | Votes: {final_votes} | Error Ratio: {final_ratio:.4f}\n')

        return final_answer

    def solve_problem(self, problem: str, ground_truth_answer: int | None = None) -> int:
        
        problem_start_time = time.time()
        self.problem_counter += 1
        problem_id = self.problem_counter
        
        print(f'\nProblem {problem_id}: {problem[:200]}...\n')

        user_input = f'{problem} {self.cfg.preference_prompt}'
        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global
        problems_left_others = max(0, self.problems_remaining - 1)
        
        # Fixed high budget = Tier 0's max_budget (always offer maximum possible)
        current_high = self.cfg.budget_tiers[0]['max_budget']
        
        # Calculate reserved time using tiered system (assumes current uses highest available tier)
        reserved_time = self._calculate_reserved_time(problems_left_others)
        
        budget = time_left - reserved_time
        budget = min(budget, current_high)
        budget = max(budget, self.cfg.base_problem_timeout)

        deadline = time.time() + budget

        print(f'Budget: {budget:.2f}s | High: {current_high}s | Reserved: {reserved_time:.1f}s | Tier slots: {self.tier_slots_remaining}\n')

        tasks = []

        for attempt_index in range(self.cfg.attempts):
            tasks.append((self.cfg.system_prompt, attempt_index))

        detailed_results = []
        valid_answers = []

        stop_event = threading.Event()

        executor = ThreadPoolExecutor(max_workers=self.cfg.workers)

        try:
            futures = []

            for (system_prompt, attempt_index) in tasks:
                future = executor.submit(
                    self._process_attempt, 
                    user_input, 
                    system_prompt, 
                    attempt_index, 
                    stop_event, 
                    deadline
                )

                futures.append(future)

            for future in as_completed(futures):
                try:
                    result = future.result()
                    detailed_results.append(result)

                    if result['Answer'] is not None:
                        valid_answers.append(result['Answer'])

                    # Wait, missing logic? early_stop logic should check majority 
                    counts = Counter(valid_answers).most_common(1)

                    if counts and counts[0][1] >= self.cfg.early_stop:
                        stop_event.set()

                        for f in futures:
                            f.cancel()

                        break

                except Exception as exc:
                    print(f'Future failed: {exc}')
                    continue

        finally:
            executor.shutdown(wait=False, cancel_futures=True)
            self.problems_remaining = max(0, self.problems_remaining - 1)

        # Record actual time usage for tiered budget tracking
        used_time = time.time() - problem_start_time
        self._record_time_usage(used_time)
        
        # Print the inference time and budget
        saved_time = max(0.0, budget - used_time)
        print(f"[Budget]: {budget:.2f}s (High was {current_high}s)\n")
        print(f"[Inference] Took {used_time:.2f}s\n")
        print(f"[Saved time]: {saved_time:.2f}s\n")

        if detailed_results:
            # Prepare display dataframe (without GroupConf, TokenConf, and FullReasoning columns)
            display_results = []
            for r in detailed_results:
                display_results.append({
                    'Attempt': r['Attempt'],
                    'Answer': r['Answer'],
                    'Response Length': r['Response Length'],
                    'Python Calls': r['Python Calls'],
                    'Python Errors': r['Python Errors']
                })
            
            results_dataframe = pd.DataFrame(display_results)
            results_dataframe['Answer'] = results_dataframe['Answer'].astype('Int64')
            display(results_dataframe)

        if not valid_answers:
            print('\nResult: 0\n')
            return 0

        final_answer = self._select_answer(detailed_results)

        return final_answer

    def __del__(self):

        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()

        if hasattr(self, 'log_file'):
            self.log_file.close()

        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                try:
                    sb = self.sandbox_pool.get_nowait()
                    sb.close()

                except Exception:
                    pass

In [15]:
solver = AIMO3Solver(CFG)

Loading model weights from /kaggle/input/gpt-oss-120b/transformers/default/1 into OS Page Cache...
Processed 26 files (65.28 GB) in 85.29 seconds.

Waiting for vLLM server...
Server is ready (took 116.73 seconds).

Initializing 16 persistent Jupyter kernels...
Kernels initialized in 2.99 seconds.



In [16]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    global correct_count, total_count, predictions
    
    question_id = id_.item(0)
    question_text = question.item(0)
    
    print("------")
    print(f"ID: {question_id}")
    print(f"Question: {question_text[:200]}...")
    
    # Get ground truth for plotting (only available in local validation)
    gt_answer = ground_truth.get(question_id, None)
    
    final_answer = solver.solve_problem(question_text, ground_truth_answer=gt_answer)
    predictions[question_id] = final_answer

    # Check accuracy if ground truth available
    total_count += 1
    if question_id in ground_truth:
        gt = ground_truth[question_id]
        is_correct = (final_answer == gt)
        if is_correct:
            correct_count += 1
        status = "✅" if is_correct else "❌"
        print(f"Answer: {final_answer} | Ground Truth: {gt} | {status}")
        print(f"📊 Running Accuracy: {correct_count}/{total_count} ({100*correct_count/total_count:.1f}%)")
    else:
        print(f"Answer: {final_answer}")
    
    print("------\n")
    
    return pl.DataFrame({'id': question_id, 'answer': final_answer})

In [17]:
# # Load reference data and keep ground truth for accuracy calculation
# df = pd.read_csv(
#     "/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv"
# )

# # Store ground truth answers for accuracy calculation (only in local mode)
# ground_truth = dict(zip(df["id"], df["answer"])) if "answer" in df.columns else {}

# # Create input file without answers
# df.drop("answer", axis=1, errors="ignore").to_csv("reference.csv", index=False)

# # Track predictions for accuracy calculation
# predictions = {}
# correct_count = 0
# total_count = 0

In [18]:
# Load reference data and keep ground truth for accuracy calculation
df = pd.read_csv(
    "/kaggle/input/omni-math-hardestdifficulty-9/omni_math_hard.csv"
)

# df = df[df['id']==12].copy()

df = df[['id','problem','answer']]

# Store ground truth answers for accuracy calculation (only in local mode)
ground_truth = dict(zip(df["id"], df["answer"])) if "answer" in df.columns else {}

# Create input file without answers
df.drop("answer", axis=1, errors="ignore").to_csv("reference.csv", index=False)

# Track predictions for accuracy calculation
predictions = {}
correct_count = 0
total_count = 0

print(f"Dataset prepared with {len(df)} problems.")

Dataset prepared with 28 problems.


In [19]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
    
else:
    inference_server.run_local_gateway(("reference.csv",))
    #inference_server.run_local_gateway(
    #    ('/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv',)
    #)

------
ID: 1741
Question: A configuration of $4027$ points in the plane is called Colombian if it consists of $2013$ red points and $2014$ blue points, and no three of the points of the configuration are collinear. By drawing ...

Problem 1: A configuration of $4027$ points in the plane is called Colombian if it consists of $2013$ red points and $2014$ blue points, and no three of the points of the configuration are collinear. By drawing ...

Budget: 820.00s | High: 820s | Reserved: 12380.0s | Tier slots: [5, 9, 6, 6]

🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Pyth

,Attempt,Answer,Response Length,Python Calls,Python Errors
0,1,90,5360,2,0
1,3,2014,9292,0,0
2,8,90,9517,1,0
3,5,63,21831,2,0
4,7,64,22372,3,0
5,6,2014,25847,5,0
6,4,90,27463,4,0
7,2,2014,34820,31,3


,Answer,Votes,Total Calls,Total Errors,Error Ratio
0,90,3,7,0,0.0000
1,2014,3,36,3,0.0833
2,63,1,2,0,0.0000
3,64,1,3,0,0.0000



Final Result: 90 | Votes: 3 | Error Ratio: 0.0000

Answer: 90 | Ground Truth: 2013 | ❌
📊 Running Accuracy: 0/1 (0.0%)
------

------
ID: 1755
Question: A [i]site[/i] is any point $(x, y)$ in the plane such that $x$ and $y$ are both positive integers less than or equal to 20.

Initially, each of the 400 sites is unoccupied. Amy and Ben take turns plac...

Problem 2: A [i]site[/i] is any point $(x, y)$ in the plane such that $x$ and $y$ are both positive integers less than or equal to 20.

Initially, each of the 400 sites is unoccupied. Amy and Ben take turns plac...

Budget: 820.00s | High: 820s | Reserved: 11830.0s | Tier slots: [5, 8, 6, 6]

🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code.

,Attempt,Answer,Response Length,Python Calls,Python Errors
0,4,100,17195,2,0
1,8,100,31318,14,3
2,7,100,38747,23,3
3,3,100,39140,22,4
4,1,100,42003,20,6
5,5,100,46158,61,10
6,6,100,45970,33,12
7,2,103,56409,38,5


...

,Answer,Votes,Total Calls,Total Errors,Error Ratio
0,7,8,95,6,0.0632



Final Result: 7 | Votes: 8 | Error Ratio: 0.0632

Answer: 7 | Ground Truth: 7 | ✅
📊 Running Accuracy: 20/27 (74.1%)
------

------
ID: 1796
Question: Let $ a_1 \equal{} 11^{11}, \, a_2 \equal{} 12^{12}, \, a_3 \equal{} 13^{13}$, and $ a_n \equal{} |a_{n \minus{} 1} \minus{} a_{n \minus{} 2}| \plus{} |a_{n \minus{} 2} \minus{} a_{n \minus{} 3}|, n \...

Problem 28: Let $ a_1 \equal{} 11^{11}, \, a_2 \equal{} 12^{12}, \, a_3 \equal{} 13^{13}$, and $ a_n \equal{} |a_{n \minus{} 1} \minus{} a_{n \minus{} 2}| \plus{} |a_{n \minus{} 2} \minus{} a_{n \minus{} 3}|, n \...

Budget: 820.00s | High: 820s | Reserved: 0.0s | Tier slots: [0, 0, 1, 2]

🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 

,Attempt,Answer,Response Length,Python Calls,Python Errors
0,7,1,3230,13,0
1,5,1,3241,15,0
2,8,1,3877,13,0
3,3,1,4216,10,0
4,4,1,4731,16,0
5,6,1,6501,18,0
6,2,1,6490,34,1
7,1,1,11525,53,2


,Answer,Votes,Total Calls,Total Errors,Error Ratio
0,1,8,172,3,0.0174



Final Result: 1 | Votes: 8 | Error Ratio: 0.0174

Answer: 1 | Ground Truth: 1 | ✅
📊 Running Accuracy: 21/28 (75.0%)
------

